Code for transforming `squad_v2` and `ftrace` to conversational format.

In [ ]:
import random
from datasets import DatasetDict, load_dataset
from huggingface_hub import whoami


username = whoami()["name"]
dataset = load_dataset("rajpurkar/squad_v2", "squad_v2")
# prompt format from [How to Train Long-Context Language Models (Effectively)](https://aclanthology.org/2025.acl-long.366/) (Gao et al., ACL 2025)

refusal_instruction = "If the context does not contain the answer, reply with: 'The question cannot be answered from this context.'"
prompts = [
    "Given the document, please answer the question.",
    "Here is a piece of text; answer the following question based on it.",
    "Please answer the question using the provided content.",
    "Based on the given passage, respond to the question.",
    "Read the snippet and answer the question that follows.",
    "Using the provided text, answer the following question."
]

templates = [
    "{prompt} {refusal_instruction}\n\n{documents}\n\nQuestion: {question}",
    "{prompt} {refusal_instruction}\n\n==== document starts ====\n{documents}\n==== document ends ====\n\nQuestion:\n{question}",
    "{prompt} {refusal_instruction}\n\n{documents}\n\n{question}",
    "{prompt} {refusal_instruction} Question: {question}\n\n{documents}",
    "{prompt} {refusal_instruction} {question}\n\n{documents}",
    "{prompt} {refusal_instruction} \n\n{question}\n\n{documents}",
]

def convert(example, idx=None):
    prompt = prompts[idx % len(prompts)]
    template = templates[(idx // len(prompts)) % len(templates)]

    user_message = template.format(
        prompt=prompt,
        documents=example["context"],
        question=example["question"],
        refusal_instruction=refusal_instruction
    )

    answer_text = (
        example["answers"]["text"][0]
        if len(example["answers"]["text"]) > 0
        else "The question cannot be answered from this context."
    )

    return {
        "id": example["id"],
        "title": example["title"],
        "context": example["context"],
        "question": example["question"],
        "answers": example["answers"],
        "messages": [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": answer_text},
        ],
        "source": "rajpurkar/squad_v2",
        "prompt_used": prompt,
    }

train_ds = dataset["train"].shuffle(seed=42).map(
    convert,
    with_indices=True,
    num_proc=1,
    remove_columns=dataset["train"].column_names
)

validation_ds = dataset["validation"].shuffle(seed=42).map(
    convert,
    with_indices=True,
    num_proc=1,
    remove_columns=dataset["validation"].column_names
)

conversational_dataset = DatasetDict({
    "train": train_ds.shuffle(seed=42),
    "validation": validation_ds.shuffle(seed=42)
})

conversational_dataset.push_to_hub(
    "squad_v2_conversational",
    private=True
)

In [ ]:
from datasets import load_dataset
from huggingface_hub import whoami
username = whoami()["name"]

dataset = load_dataset("DataAttributionEval/ftrace", "Llama-3.1-8B")
def convert(example):
    return {
        "id": example["data_id"],
        "messages": [
            {"role": "user", "content": example["prompt"]},
            {"role": "assistant", "content": example["response"]},
        ],
        "source": "DataAttributionEval/ftrace",
        "facts": example["facts"]
    }

conversational_dataset = dataset["train"].map(
    convert,
    remove_columns=dataset["train"].column_names
)

conversational_dataset.push_to_hub(
    "ftrace_date-lm_conversational",
    split="train",
    private=True
)